In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach ['p;oKaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv
/kaggle/input/models/shree0910/minilm-sentence-transformer/pytorch/base/1/config.json
/kaggle/input/models/shree0910/minilm-sentence-transformer/pytorch/base/1/README.md
/kaggle/input/models/shree0910/minilm-sentence-transformer/pytorch/base/1/tokenizer.json
/kaggle/input/models/shree0910/min

In [2]:
 #Load and explore data: corpus to search, labeled training questions, ground-truth relevance grades (0-3), and unlabeled test questions
documents = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv")
train_queries = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv")
qrels_train = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv")
test_queries = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv")

print("documents:", documents.shape)
print("train_queries:", train_queries.shape)
print("qrels_train:", qrels_train.shape)
print("test_queries:", test_queries.shape)

print("\nrelevance value counts:")
print(qrels_train["relevance"].value_counts())

train_queries.head()

documents: (695, 9)
train_queries: (308, 3)
qrels_train: (4194, 3)
test_queries: (200, 2)

relevance value counts:
relevance
0.0    2724
1.0     728
3.0     556
2.0     186
Name: count, dtype: int64


,query_id,query,positive_docs
0,1,How do I cope with flooding and excess rain on...,8 9 10 7 6
1,2,How can I adapt my farming to flooding and exc...,8 9 10 7 6
2,3,How does flooding and excess rain affect my cr...,6 8 9 10 7
3,4,What is the risk of flooding and excess rain t...,6 8 9 10 7
4,5,How do I manage bacterial leaf blight in rice?,40 36 38


In [3]:
q5_qrels = qrels_train[qrels_train["query_id"] == 5].merge(
    documents[["document_id", "title", "crop"]], on="document_id"
)
q5_qrels.sort_values("relevance", ascending=False)

,query_id,document_id,relevance,title,crop
0,5,40,3.0,Managing Bacterial Leaf Blight in Rice (Irriga...,Rice
1,5,36,3.0,Managing Bacterial Leaf Blight in Rice (Guinea...,Rice
2,5,38,3.0,Managing Bacterial Leaf Blight in Rice (Humid ...,Rice
3,5,34,0.0,How Bacterial Leaf Blight spreads in Rice,Rice
4,5,35,0.0,Symptoms of Bacterial Leaf Blight in Rice,Rice
5,5,37,0.0,Preventing Bacterial Leaf Blight in Rice (Guin...,Rice
6,5,39,0.0,Preventing Bacterial Leaf Blight in Rice (Humi...,Rice
7,5,41,0.0,Preventing Bacterial Leaf Blight in Rice (Irri...,Rice
8,5,139,0.0,Managing Rice Blast in Rice (Irrigated lowland),Rice
9,5,140,0.0,Preventing Rice Blast in Rice (Irrigated lowland),Rice


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
doc_mat = vec.fit_transform(documents["title"] + ". " + documents["text"])

query_text = train_queries.loc[train_queries["query_id"] == 5, "query"].iloc[0]
print("Query:", query_text)

q_vec = vec.transform([query_text])
sims = cosine_similarity(q_vec, doc_mat)[0]

result = documents[["document_id", "title"]].copy()
result["score"] = sims

q5_labels = qrels_train[qrels_train["query_id"] == 5][["document_id", "relevance"]]
result = result.merge(q5_labels, on="document_id", how="left")
result["relevance"] = result["relevance"].fillna(0)

result.sort_values("score", ascending=False).head(15)

Query: How do I manage bacterial leaf blight in rice?


,document_id,title,score,relevance
34,35,Symptoms of Bacterial Leaf Blight in Rice,0.591765,0.0
33,34,How Bacterial Leaf Blight spreads in Rice,0.557796,0.0
36,37,Preventing Bacterial Leaf Blight in Rice (Guin...,0.513954,0.0
38,39,Preventing Bacterial Leaf Blight in Rice (Humi...,0.496173,0.0
35,36,Managing Bacterial Leaf Blight in Rice (Guinea...,0.478839,3.0
37,38,Managing Bacterial Leaf Blight in Rice (Humid ...,0.464389,3.0
40,41,Preventing Bacterial Leaf Blight in Rice (Irri...,0.452995,0.0
39,40,Managing Bacterial Leaf Blight in Rice (Irriga...,0.451993,3.0
150,151,Management of Rice Blast and Bacterial Leaf Bl...,0.326228,0.0
654,655,Enhancing Rice Defenses with Soil Silicon,0.235644,0.0


In [5]:
# Evaluation harness (nDCG@5) 
#Ground-truth lookup: query_id -> {document_id: relevance grade}. Docs not listed here count as relevance 0.

import numpy as np

# Lookup: query_id -> {document_id: relevance}
qrels_lookup = qrels_train.groupby("query_id")[["document_id", "relevance"]].apply(
    lambda df: dict(zip(df["document_id"], df["relevance"]))
).to_dict()

def ndcg_at_k(ranked_doc_ids, relevance_map, k=5):
    relevances = [relevance_map.get(doc_id, 0.0) for doc_id in ranked_doc_ids[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))

    ideal_relevances = sorted(relevance_map.values(), reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevances))

    return dcg / idcg if idcg > 0 else 0.0

# Score the TF-IDF baseline across all training queries
q_mat_train = vec.transform(train_queries["query"])
sims_train = cosine_similarity(q_mat_train, doc_mat)

doc_ids_array = documents["document_id"].values

scores = []
for i, qid in enumerate(train_queries["query_id"]):
    top5_idx = sims_train[i].argsort()[::-1][:5]
    top5_doc_ids = doc_ids_array[top5_idx]
    relevance_map = qrels_lookup.get(qid, {})
    scores.append(ndcg_at_k(top5_doc_ids, relevance_map, k=5))

print("Mean nDCG@5 (TF-IDF baseline, train set):", np.mean(scores))

Mean nDCG@5 (TF-IDF baseline, train set): 0.5030820195615423


In [6]:
# # Same lexical idea as TF-IDF, plus two corrections: diminishing returns for repeated terms,
# and normalizing for document length so long docs aren't favored just for containing more words.

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

cv = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
doc_term = cv.fit_transform(documents["title"] + ". " + documents["text"])  # raw counts, (n_docs, n_terms)
doc_term_csc = doc_term.tocsc()  # fast column access per term

doc_lengths = np.asarray(doc_term.sum(axis=1)).ravel()
avgdl = doc_lengths.mean()

n_docs = doc_term.shape[0]
df = np.asarray((doc_term > 0).sum(axis=0)).ravel()
idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)

k1, b = 1.5, 0.75  # standard BM25 defaults: k1 = term-frequency saturation, b = length-normalization strength

def bm25_scores_for_query(query_text):
    q_terms = cv.transform([query_text]).indices  # vocab indices present in the query
    scores = np.zeros(n_docs)
    for t in q_terms:
        tf = np.asarray(doc_term_csc[:, t].todense()).ravel()
        denom = tf + k1 * (1 - b + b * doc_lengths / avgdl)
        scores += idf[t] * (tf * (k1 + 1)) / np.where(denom == 0, 1, denom)
    return scores

scores_bm25 = []
for i, qid in enumerate(train_queries["query_id"]):
    scores = bm25_scores_for_query(train_queries.iloc[i]["query"])
    top5_idx = np.argsort(scores)[::-1][:5]
    top5_doc_ids = doc_ids_array[top5_idx]
    relevance_map = qrels_lookup.get(qid, {})
    scores_bm25.append(ndcg_at_k(top5_doc_ids, relevance_map, k=5))

print("Mean nDCG@5 (BM25, train set):", np.mean(scores_bm25))

Mean nDCG@5 (BM25, train set): 0.5218982644152164


In [7]:
scores_q5_bm25 = bm25_scores_for_query(train_queries.loc[train_queries["query_id"] == 5, "query"].iloc[0])
result_bm25 = documents[["document_id", "title"]].copy()
result_bm25["score"] = scores_q5_bm25
result_bm25 = result_bm25.merge(qrels_train[qrels_train["query_id"] == 5][["document_id", "relevance"]], on="document_id", how="left")
result_bm25["relevance"] = result_bm25["relevance"].fillna(0)
result_bm25.sort_values("score", ascending=False).head(8)

,document_id,title,score,relevance
34,35,Symptoms of Bacterial Leaf Blight in Rice,37.743175,0.0
33,34,How Bacterial Leaf Blight spreads in Rice,36.296007,0.0
40,41,Preventing Bacterial Leaf Blight in Rice (Irri...,32.403905,0.0
39,40,Managing Bacterial Leaf Blight in Rice (Irriga...,31.957974,3.0
150,151,Management of Rice Blast and Bacterial Leaf Bl...,31.913129,0.0
36,37,Preventing Bacterial Leaf Blight in Rice (Guin...,30.451250,0.0
35,36,Managing Bacterial Leaf Blight in Rice (Guinea...,29.935934,3.0
38,39,Preventing Bacterial Leaf Blight in Rice (Humi...,29.635031,0.0


In [8]:
# Semantic retrieval: a transformer encoder understands word RELATIONSHIPS via self-attention, not just
# word identity, this is what lets it separate "manage" from "prevent" where BM25/TF-IDF couldn't.
# Loaded via raw transformers (not the SentenceTransformer wrapper) due to a library version mismatch;
# this mean pooling is done by hand below to turn per-token vectors into one vector per document/query.

import torch
from transformers import AutoTokenizer, AutoModel

model_path = "/kaggle/input/models/shree0910/minilm-sentence-transformer/pytorch/base/1"

tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModel.from_pretrained(model_path)
base_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch, seq_len, hidden)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
        with torch.no_grad():
            output = base_model(**encoded)
        pooled = mean_pooling(output, encoded["attention_mask"])
        normed = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(normed)
    return torch.cat(all_embeddings, dim=0).numpy()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
# Score dense retrieval
doc_texts = (documents["title"] + ". " + documents["text"]).tolist()
doc_embeddings = embed_texts(doc_texts, batch_size=32)

query_embeddings = embed_texts(train_queries["query"].tolist(), batch_size=32)

sims_dense = query_embeddings @ doc_embeddings.T  # dot product == cosine sim, since both are L2-normalized

scores_dense = []
for i, qid in enumerate(train_queries["query_id"]):
    top5_idx = np.argsort(sims_dense[i])[::-1][:5]
    top5_doc_ids = doc_ids_array[top5_idx]
    relevance_map = qrels_lookup.get(qid, {})
    scores_dense.append(ndcg_at_k(top5_doc_ids, relevance_map, k=5))

print("Mean nDCG@5 (dense embeddings, train set):", np.mean(scores_dense))

Mean nDCG@5 (dense embeddings, train set): 0.7082923572364125


In [10]:
q5_idx = train_queries.index[train_queries["query_id"] == 5][0]
top_idx = np.argsort(sims_dense[q5_idx])[::-1][:8]

result_dense = documents.iloc[top_idx][["document_id", "title"]].copy()
result_dense["score"] = sims_dense[q5_idx][top_idx]
result_dense = result_dense.merge(qrels_train[qrels_train["query_id"] == 5][["document_id", "relevance"]], on="document_id", how="left")
result_dense["relevance"] = result_dense["relevance"].fillna(0)
result_dense

,document_id,title,score,relevance
0,40,Managing Bacterial Leaf Blight in Rice (Irriga...,0.813373,3.0
1,38,Managing Bacterial Leaf Blight in Rice (Humid ...,0.800938,3.0
2,41,Preventing Bacterial Leaf Blight in Rice (Irri...,0.793648,0.0
3,39,Preventing Bacterial Leaf Blight in Rice (Humi...,0.790587,0.0
4,35,Symptoms of Bacterial Leaf Blight in Rice,0.757927,0.0
5,34,How Bacterial Leaf Blight spreads in Rice,0.754774,0.0
6,36,Managing Bacterial Leaf Blight in Rice (Guinea...,0.741354,3.0
7,37,Preventing Bacterial Leaf Blight in Rice (Guin...,0.715576,0.0


In [11]:
from transformers import AutoModelForSequenceClassification

ce_path = "/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2"
ce_tokenizer = AutoTokenizer.from_pretrained(ce_path)
ce_model = AutoModelForSequenceClassification.from_pretrained(ce_path)
ce_model.eval()

doc_text_lookup = dict(zip(documents["document_id"], documents["title"] + ". " + documents["text"]))

def rerank(query_text, candidate_doc_ids, batch_size=16):
    # cross-encoder: query and doc are encoded TOGETHER, so attention can directly compare their tokens
    pairs = [doc_text_lookup[d] for d in candidate_doc_ids]
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch_docs = pairs[i:i + batch_size]
        encoded = ce_tokenizer(
            [query_text] * len(batch_docs), batch_docs,
            padding=True, truncation=True, max_length=256, return_tensors="pt"
        )
        with torch.no_grad():
            logits = ce_model(**encoded).logits.squeeze(-1)
        scores.extend(logits.tolist())
    return scores

SHORTLIST_K = 20  # how many dense-retrieval candidates the cross-encoder gets to re-rank

scores_rerank = []
for i, qid in enumerate(train_queries["query_id"]):
    query_text = train_queries.iloc[i]["query"]
    shortlist_idx = np.argsort(sims_dense[i])[::-1][:SHORTLIST_K]
    shortlist_doc_ids = doc_ids_array[shortlist_idx]

    ce_scores = rerank(query_text, shortlist_doc_ids)
    order = np.argsort(ce_scores)[::-1][:5]
    top5_doc_ids = shortlist_doc_ids[order]

    scores_rerank.append(ndcg_at_k(top5_doc_ids, qrels_lookup.get(qid, {}), k=5))

print("Mean nDCG@5 (dense + cross-encoder rerank, train set):", np.mean(scores_rerank))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mean nDCG@5 (dense + cross-encoder rerank, train set): 0.777963711493539


In [12]:
q5_idx = train_queries.index[train_queries["query_id"] == 5][0]
q5_text = train_queries.loc[q5_idx, "query"]

shortlist_idx = np.argsort(sims_dense[q5_idx])[::-1][:SHORTLIST_K]
shortlist_doc_ids = doc_ids_array[shortlist_idx]
ce_scores_q5 = rerank(q5_text, shortlist_doc_ids)

result_rerank = documents[documents["document_id"].isin(shortlist_doc_ids)][["document_id", "title"]].copy()
score_map = dict(zip(shortlist_doc_ids, ce_scores_q5))
result_rerank["ce_score"] = result_rerank["document_id"].map(score_map)
result_rerank = result_rerank.merge(qrels_train[qrels_train["query_id"] == 5][["document_id", "relevance"]], on="document_id", how="left")
result_rerank["relevance"] = result_rerank["relevance"].fillna(0)
result_rerank.sort_values("ce_score", ascending=False).head(8)

,document_id,title,ce_score,relevance
6,40,Managing Bacterial Leaf Blight in Rice (Irriga...,9.028123,3.0
4,38,Managing Bacterial Leaf Blight in Rice (Humid ...,8.932710,3.0
2,36,Managing Bacterial Leaf Blight in Rice (Guinea...,8.647354,3.0
7,41,Preventing Bacterial Leaf Blight in Rice (Irri...,8.314846,0.0
5,39,Preventing Bacterial Leaf Blight in Rice (Humi...,7.553618,0.0
3,37,Preventing Bacterial Leaf Blight in Rice (Guin...,7.114336,0.0
15,151,Management of Rice Blast and Bacterial Leaf Bl...,5.638914,0.0
0,34,How Bacterial Leaf Blight spreads in Rice,4.438580,0.0


In [13]:
#Submission
test_embeddings = embed_texts(test_queries["query"].tolist())
sims_test = test_embeddings @ doc_embeddings.T

rows = []
for i, qid in enumerate(test_queries["query_id"]):
    query_text = test_queries.iloc[i]["query"]
    shortlist_idx = np.argsort(sims_test[i])[::-1][:SHORTLIST_K]
    shortlist_doc_ids = doc_ids_array[shortlist_idx]

    ce_scores = rerank(query_text, shortlist_doc_ids)
    order = np.argsort(ce_scores)[::-1][:5]
    top5_doc_ids = shortlist_doc_ids[order]

    for doc_id in top5_doc_ids:
        rows.append({"QueryId": qid, "DocumentId": int(doc_id)})

submission = pd.DataFrame(rows)

assert submission.groupby("QueryId").size().eq(5).all(), "every query must have exactly 5 rows"
assert set(submission["QueryId"]) == set(test_queries["query_id"]), "every test query must appear"

submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head(10)

(1000, 2)


,QueryId,DocumentId
0,1001,4
1,1001,1
2,1001,3
3,1001,5
4,1001,2
5,1002,4
6,1002,3
7,1002,5
8,1002,2
9,1002,1
